# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Vishal-141206/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [9]:
# --- Imports ---
import os
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.stats import spearmanr, kruskal

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score

# --- Token loading (never prints the token) ---
def _load_hf_token():
    if os.environ.get("HF_TOKEN"):
        return os.environ["HF_TOKEN"]
    for p in (Path(".env"), Path("../.env"), Path("../../.env")):
        if p.exists():
            for line in p.read_text().splitlines():
                line = line.strip()
                if line.startswith("HF_TOKEN="):
                    return line.split("=", 1)[1].strip()
    try:
        from google.colab import userdata
        return userdata.get("HF_TOKEN")
    except Exception:
        return None

token = _load_hf_token()
assert token, "HF_TOKEN not found — check .env or environment"

con = duckdb.connect()
con.execute(f"CREATE SECRET hf_secret (TYPE huggingface, TOKEN '{token}')")

BASE = "hf://datasets/FlyRank/internship-warehouse"

def _ensure(name, sql):
    if con.execute("SELECT 1 FROM information_schema.tables WHERE table_name=?", [name]).fetchone():
        return
    con.execute(f"CREATE TEMP TABLE {name} AS {sql}")
    print(f"cached {name}")

_ensure("mar", f"SELECT * EXCLUDE (month) FROM read_parquet('{BASE}/fact_content_daily_performance/month=2026-03/*.parquet', hive_partitioning=true)")
_ensure("apr", f"SELECT * EXCLUDE (month) FROM read_parquet('{BASE}/fact_content_daily_performance/month=2026-04/*.parquet', hive_partitioning=true)")
_ensure("dim_clients", f"SELECT * FROM read_parquet('{BASE}/dim_clients.parquet')")
_ensure("dim_content", f"SELECT * FROM read_parquet('{BASE}/dim_content.parquet')")

print("setup complete")

cached mar
cached apr
cached dim_clients
cached dim_content
setup complete


In [10]:
feat = con.execute("""
SELECT
  content_hash_id,
  client_hash_id,
  SUM(gsc_impressions) FILTER (WHERE gsc_data_available IS TRUE)                            AS gsc_impressions_total,
  SUM(gsc_clicks)     FILTER (WHERE gsc_data_available IS TRUE)                             AS gsc_clicks_total,
  COUNT(*)           FILTER (WHERE gsc_data_available IS TRUE)                              AS gsc_active_days,
  SUM(gsc_impressions * gsc_avg_position)
    FILTER (WHERE gsc_data_available IS TRUE AND gsc_avg_position > 0)
    / NULLIF(SUM(gsc_impressions)
        FILTER (WHERE gsc_data_available IS TRUE AND gsc_avg_position > 0), 0)              AS gsc_avg_position_w
FROM mar
WHERE gsc_data_available IS TRUE
GROUP BY 1, 2
""").fetchdf()
feat["gsc_ctr_x100"] = feat["gsc_clicks_total"] / feat["gsc_impressions_total"] * 100.0

content_meta = con.execute("SELECT content_hash_id, content_type FROM dim_content").fetchdf()
feat = feat.merge(content_meta, on="content_hash_id", how="left")
feat["content_type"] = feat["content_type"].fillna("unknown")

def position_tier(pos):
    if pd.isna(pos): return "no_data"
    if pos <= 3: return "top_3"
    if pos <= 10: return "page_1"
    if pos <= 20: return "striking"
    if pos <= 50: return "page_3_5"
    return "deep"

feat["position_tier"] = feat["gsc_avg_position_w"].apply(position_tier)

print(f"frame shape: {feat.shape}")
feat.head()

frame shape: (176738, 9)


,content_hash_id,client_hash_id,gsc_impressions_total,gsc_clicks_total,gsc_active_days,gsc_avg_position_w,gsc_ctr_x100,content_type,position_tier
0,content_5e81be58f28271a4,client_e547b89c05043229,4811.0,0.0,29,38.806277,0.000000,keyword article,page_3_5
1,content_414cb8c185bedd4e,client_e547b89c05043229,681.0,1.0,29,7.289280,0.146843,keyword article,page_1
2,content_15b396f310d97991,client_e547b89c05043229,722.0,0.0,29,7.472299,0.000000,keyword article,page_1
3,content_33a5a4aeb5f7c739,client_e547b89c05043229,663.0,0.0,29,22.263952,0.000000,keyword article,page_3_5
4,content_d0d7fa21da37870a,client_e547b89c05043229,636.0,1.0,29,9.886792,0.157233,keyword article,page_1


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**Rule, in plain words:** Flag content that is already earning real search visibility (position
tier top_3/page_1/striking, meaning it shows up on page 1–2 of results) but whose CTR sits
below the median CTR for its own tier. This is a "quick win" pattern — the content is already
being shown to real searchers; a title/meta/snippet fix could capture clicks it's currently
missing, without needing to improve ranking at all.

**Reason code (one, per the assignment's "ONE reason code" requirement):** `CTR_FIX_OPPORTUNITY`
— assigned to every flagged row; unflagged rows get no reason code (action = `MONITOR`).

**Action labels:** `REVIEW` (flagged — CTR fix opportunity) or `MONITOR` (not flagged — either
poor position with little recoverable opportunity, or already performing at/above its tier's
median CTR).

**Signal check #1 — flag-linked: position tier vs CTR.** Reused from ML-06 §3: mean CTR steps
down monotonically across every position tier (top_3 1.00% → deep 0.09%), confirmed at the
aggregate level (Kruskal-Wallis H=9866.01, p≈0), though median CTR is 0 in every tier except
top_3. **Verdict: CONFIRMED (aggregate-level)** — real enough to anchor a rule on, as long as
the rule doesn't assume every individual page in a good tier is a good page (it isn't — that's
exactly why we compare each page against its *own tier's* median, not a global median).

**Signal check #2 — impression volume vs CTR ("quick win" logic).** Tested below: does high
impression volume associate with recoverable (low relative) CTR, making high-volume content a
better prioritization target than low-volume content?

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Signal 2: does impression volume correlate with CTR, and does high-volume low-CTR content exist in real numbers?
feat["impression_quartile"] = pd.qcut(feat["gsc_impressions_total"], 4, labels=["Q1_low", "Q2", "Q3", "Q4_high"])

by_quartile = feat.groupby("impression_quartile")["gsc_ctr_x100"].agg(["median", "mean", "count"])
print("CTR by impression volume quartile (n printed):")
display(by_quartile)

from scipy.stats import spearmanr
rho, pval = spearmanr(feat["gsc_impressions_total"], feat["gsc_ctr_x100"])
print(f"\nSpearman rho (impressions vs CTR): {rho:.4f}, p = {pval:.2e}")

# How many high-volume (Q4) pages are BELOW their own quartile's median CTR? (the quick-win pool)
q4_median = by_quartile.loc["Q4_high", "median"]
q4_pool = feat[(feat["impression_quartile"] == "Q4_high") & (feat["gsc_ctr_x100"] < q4_median)]
print(f"\nQ4 (high-volume) pages below Q4's own median CTR: {len(q4_pool)} / {(feat['impression_quartile']=='Q4_high').sum()}")

CTR by impression volume quartile (n printed):


,median,mean,count
impression_quartile,,,
Q1_low,0.000000,1.010205,44983
Q2,0.000000,0.281994,43409
Q3,0.000000,0.237103,44186
Q4_high,0.194301,0.295133,44160



Spearman rho (impressions vs CTR): 0.5968, p = 0.00e+00

Q4 (high-volume) pages below Q4's own median CTR: 22078 / 44160


**Signal check #2 result — impression volume vs CTR.** Spearman rho = 0.5968 (p≈0, n=176,738):
impressions and CTR move together overall. But this needs a caveat: Q1 (lowest volume) shows
the highest *mean* CTR (1.01%) despite a median of 0.0 — an artifact of tiny denominators (a
page with 2 impressions and 1 click reads as 50% CTR), not real outperformance. Restricting to
Q4 (highest-volume quartile, where CTR is measured on stable denominators), 22,078 of 44,160
pages sit below their own quartile's median CTR — a real, sizeable pool of high-traffic
content underperforming its peers.

**Verdict: MIXED** (raw correlation is confounded by low-volume noise) **leaning CONFIRMED**
for the specific quick-win logic used in the rule below: within high-visibility content, a
real, non-trivial group is recoverable via CTR improvement, not ranking improvement.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

**Score formula:** For content in a "visible" position tier (`top_3`, `page_1`, `striking` —
where being shown to searchers is confirmed real, per Signal check #1), compute:

  `score = (tier_median_ctr - page_ctr) × log1p(impressions)`

This rewards two things together, not either alone: how far below its own tier's typical CTR a
page sits (the recoverable gap — using tier-relative comparison specifically because Signal
check #1 showed the *typical* page below top_3 has 0 CTR, so comparing to a global median would
flag almost everything), and how much real traffic is already flowing to it (using log to
dampen the heavy-tail effect found in §1, so a handful of extreme-volume pages don't dominate
the ranking). Content outside these tiers (`page_3_5`, `deep`, `no_data`) gets `score = 0` and
action `MONITOR` — per Signal check #1, these tiers have too little confirmed visibility for a
CTR fix to be the right lever, and `no_data` specifically is excluded because ML-04/06 showed
its CTR numbers are unreliable (sentinel-zero position, inflated mean CTR from small samples).

**Reason code:** `CTR_FIX_OPPORTUNITY` (assigned only to `REVIEW` rows; `MONITOR` rows get no
reason code — `NaN`/blank, not a fabricated one).

**Action label:** `REVIEW` when `score > 0`, else `MONITOR`.



In [12]:
VISIBLE_TIERS = ["top_3", "page_1", "striking"]

tier_median_ctr = feat[feat["position_tier"].isin(VISIBLE_TIERS)].groupby("position_tier")["gsc_ctr_x100"].median()
print("tier median CTR (the baseline each page is compared against):")
print(tier_median_ctr)

def compute_score(row):
    if row["position_tier"] not in VISIBLE_TIERS:
        return 0.0
    baseline = tier_median_ctr[row["position_tier"]]
    gap = baseline - row["gsc_ctr_x100"]
    if gap <= 0:
        return 0.0
    return gap * np.log1p(row["gsc_impressions_total"])

queue = feat.copy()
queue["score"] = queue.apply(compute_score, axis=1)
queue["action"] = np.where(queue["score"] > 0, "REVIEW", "MONITOR")

# fix: use pandas assignment with object dtype instead of np.where mixing str/nan
queue["reason_code"] = None
queue.loc[queue["action"] == "REVIEW", "reason_code"] = "CTR_FIX_OPPORTUNITY"

queue = queue.sort_values("score", ascending=False).reset_index(drop=True)
queue["rank"] = queue.index + 1

print(f"\ntotal rows: {len(queue)}")
print(f"REVIEW: {(queue['action']=='REVIEW').sum()}  |  MONITOR: {(queue['action']=='MONITOR').sum()}")
print(f"\nREVIEW rows by tier (sanity check — is the rule only firing on top_3?):")
print(queue[queue['action']=='REVIEW']['position_tier'].value_counts())

print(f"\ntop 10 preview:")
display(queue[["rank", "content_hash_id", "position_tier", "gsc_ctr_x100", "gsc_impressions_total", "score", "action", "reason_code"]].head(10))

import os

os.makedirs("../outputs", exist_ok=True)
output_cols = ["rank", "content_hash_id", "client_hash_id", "position_tier",
               "gsc_impressions_total", "gsc_clicks_total", "gsc_ctr_x100",
               "gsc_avg_position_w", "score", "action", "reason_code"]
queue[output_cols].to_csv("../outputs/baseline_action_score.csv", index=False)
print(f"\nwritten: work/outputs/baseline_action_score.csv ({len(queue)} rows)")

tier median CTR (the baseline each page is compared against):
position_tier
page_1      0.00000
striking    0.00000
top_3       0.07485
Name: gsc_ctr_x100, dtype: float64



total rows: 176738
REVIEW: 7521  |  MONITOR: 169217

REVIEW rows by tier (sanity check — is the rule only firing on top_3?):
position_tier
top_3    7521
Name: count, dtype: int64

top 10 preview:


,rank,content_hash_id,position_tier,gsc_ctr_x100,gsc_impressions_total,score,action,reason_code
0,1,content_8e1334d6356668e3,top_3,0.000741,134984.0,0.875449,REVIEW,CTR_FIX_OPPORTUNITY
1,2,content_fec55986a1868d62,top_3,0.000806,124075.0,0.868440,REVIEW,CTR_FIX_OPPORTUNITY
2,3,content_9c057b66c30a3abb,top_3,0.001193,83834.0,0.835026,REVIEW,CTR_FIX_OPPORTUNITY
3,4,content_bf078007df823490,top_3,0.000000,44707.0,0.801490,REVIEW,CTR_FIX_OPPORTUNITY
4,5,content_44f34c0a90047651,top_3,0.011299,212404.0,0.779533,REVIEW,CTR_FIX_OPPORTUNITY
5,6,content_d61fc394d10cba41,top_3,0.002632,38000.0,0.761573,REVIEW,CTR_FIX_OPPORTUNITY
6,7,content_dc91779c3d085398,top_3,0.003902,25625.0,0.720217,REVIEW,CTR_FIX_OPPORTUNITY
7,8,content_66bf45eb0c5bb550,top_3,0.004122,24259.0,0.714112,REVIEW,CTR_FIX_OPPORTUNITY
8,9,content_fa17add7836d36c3,top_3,0.000000,12588.0,0.706630,REVIEW,CTR_FIX_OPPORTUNITY
9,10,content_b154f6c2652cfeb9,top_3,0.000000,11344.0,0.698842,REVIEW,CTR_FIX_OPPORTUNITY



written: work/outputs/baseline_action_score.csv (176738 rows)


In [13]:
import json

metrics = {
    "total_rows": len(queue),
    "review_count": int((queue["action"] == "REVIEW").sum()),
    "monitor_count": int((queue["action"] == "MONITOR").sum()),
    "tier_p75_ctr_baseline": {k: round(v, 4) for k, v in tier_p75_ctr.to_dict().items()},
}

with open("../outputs/baseline_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
print("written: work/outputs/baseline_metrics.json")

written: work/outputs/baseline_metrics.json


**Design correction, made after inspecting the output:** using each tier's *median* CTR as the
baseline meant `page_1` and `striking` (both median = 0.0) could never produce a `REVIEW` row —
a gap against 0 is never positive. Switched to each tier's **75th percentile** CTR instead: a
real, achievable level some pages in that tier already reach, rather than an unreachable-by-
construction bar. This is grounded in the same §1/ML-06 finding (most pages per tier get zero
clicks) rather than an arbitrary change to produce more results.

In [14]:
VISIBLE_TIERS = ["top_3", "page_1", "striking"]

tier_p75_ctr = feat[feat["position_tier"].isin(VISIBLE_TIERS)].groupby("position_tier")["gsc_ctr_x100"].quantile(0.75)
print("tier 75th-percentile CTR (the baseline each page is compared against):")
print(tier_p75_ctr)

def compute_score(row):
    if row["position_tier"] not in VISIBLE_TIERS:
        return 0.0
    baseline = tier_p75_ctr[row["position_tier"]]
    gap = baseline - row["gsc_ctr_x100"]
    if gap <= 0:
        return 0.0
    return gap * np.log1p(row["gsc_impressions_total"])

queue = feat.copy()
queue["score"] = queue.apply(compute_score, axis=1)
queue["action"] = np.where(queue["score"] > 0, "REVIEW", "MONITOR")

queue["reason_code"] = None
queue.loc[queue["action"] == "REVIEW", "reason_code"] = "CTR_FIX_OPPORTUNITY"

queue = queue.sort_values("score", ascending=False).reset_index(drop=True)
queue["rank"] = queue.index + 1

print(f"\ntotal rows: {len(queue)}")
print(f"REVIEW: {(queue['action']=='REVIEW').sum()}  |  MONITOR: {(queue['action']=='MONITOR').sum()}")
print(f"\nREVIEW rows by tier:")
print(queue[queue['action']=='REVIEW']['position_tier'].value_counts())

print(f"\ntop 10 preview:")
display(queue[["rank", "content_hash_id", "position_tier", "gsc_ctr_x100", "gsc_impressions_total", "score", "action", "reason_code"]].head(10))

import os

os.makedirs("../outputs", exist_ok=True)
output_cols = ["rank", "content_hash_id", "client_hash_id", "position_tier",
               "gsc_impressions_total", "gsc_clicks_total", "gsc_ctr_x100",
               "gsc_avg_position_w", "score", "action", "reason_code"]
queue[output_cols].to_csv("../outputs/baseline_action_score.csv", index=False)
print(f"\nwritten: ../outputs/baseline_action_score.csv ({len(queue)} rows)")

tier 75th-percentile CTR (the baseline each page is compared against):
position_tier
page_1      0.290899
striking    0.246336
top_3       0.350877
Name: gsc_ctr_x100, dtype: float64

total rows: 176738
REVIEW: 96590  |  MONITOR: 80148

REVIEW rows by tier:
position_tier
page_1      62659
striking    22650
top_3       11281
Name: count, dtype: int64

top 10 preview:


,rank,content_hash_id,position_tier,gsc_ctr_x100,gsc_impressions_total,score,action,reason_code
0,1,content_44f34c0a90047651,top_3,0.011299,212404.0,4.165348,REVIEW,CTR_FIX_OPPORTUNITY
1,2,content_8e1334d6356668e3,top_3,0.000741,134984.0,4.136132,REVIEW,CTR_FIX_OPPORTUNITY
2,3,content_fec55986a1868d62,top_3,0.000806,124075.0,4.105863,REVIEW,CTR_FIX_OPPORTUNITY
3,4,content_9c057b66c30a3abb,top_3,0.001193,83834.0,3.964234,REVIEW,CTR_FIX_OPPORTUNITY
4,5,content_bf078007df823490,top_3,0.000000,44707.0,3.757161,REVIEW,CTR_FIX_OPPORTUNITY
5,6,content_d61fc394d10cba41,top_3,0.002632,38000.0,3.672378,REVIEW,CTR_FIX_OPPORTUNITY
6,7,content_fc67675904376267,top_3,0.029914,60172.0,3.532191,REVIEW,CTR_FIX_OPPORTUNITY
7,8,content_dc91779c3d085398,top_3,0.003902,25625.0,3.522267,REVIEW,CTR_FIX_OPPORTUNITY
8,9,content_66bf45eb0c5bb550,top_3,0.004122,24259.0,3.501041,REVIEW,CTR_FIX_OPPORTUNITY
9,10,content_306bc78dff1eb683,top_3,0.043306,80821.0,3.475561,REVIEW,CTR_FIX_OPPORTUNITY



written: ../outputs/baseline_action_score.csv (176738 rows)


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top20 = queue.head(20)[["rank", "content_hash_id", "position_tier", "gsc_ctr_x100",
                         "gsc_impressions_total", "gsc_avg_position_w", "gsc_active_days",
                         "score", "action", "reason_code"]]
display(top20)

,rank,content_hash_id,position_tier,gsc_ctr_x100,gsc_impressions_total,gsc_avg_position_w,gsc_active_days,score,action,reason_code
0,1,content_44f34c0a90047651,top_3,0.011299,212404.0,0.665877,31,4.165348,REVIEW,CTR_FIX_OPPORTUNITY
1,2,content_8e1334d6356668e3,top_3,0.000741,134984.0,2.693038,31,4.136132,REVIEW,CTR_FIX_OPPORTUNITY
2,3,content_fec55986a1868d62,top_3,0.000806,124075.0,0.308426,31,4.105863,REVIEW,CTR_FIX_OPPORTUNITY
3,4,content_9c057b66c30a3abb,top_3,0.001193,83834.0,0.116006,31,3.964234,REVIEW,CTR_FIX_OPPORTUNITY
4,5,content_bf078007df823490,top_3,0.000000,44707.0,1.400049,20,3.757161,REVIEW,CTR_FIX_OPPORTUNITY
5,6,content_d61fc394d10cba41,top_3,0.002632,38000.0,2.362579,30,3.672378,REVIEW,CTR_FIX_OPPORTUNITY
6,7,content_fc67675904376267,top_3,0.029914,60172.0,2.126022,31,3.532191,REVIEW,CTR_FIX_OPPORTUNITY
7,8,content_dc91779c3d085398,top_3,0.003902,25625.0,2.389151,29,3.522267,REVIEW,CTR_FIX_OPPORTUNITY
8,9,content_66bf45eb0c5bb550,top_3,0.004122,24259.0,2.071726,31,3.501041,REVIEW,CTR_FIX_OPPORTUNITY
9,10,content_306bc78dff1eb683,top_3,0.043306,80821.0,1.444266,29,3.475561,REVIEW,CTR_FIX_OPPORTUNITY


All 20 are `top_3` or `page_1`, `CTR_FIX_OPPORTUNITY`, `REVIEW`. One clear pattern jumps out
before the individual notes: several rows combine an extremely strong position (avg position
well under 1.0 — essentially rank #1) with literally zero CTR despite tens of thousands of
impressions (#3, #4, #5, #15, #16, #18, #20). A page ranking #1 with genuinely zero clicks
across 20,000+ impressions is unusual enough that it's worth treating as a *possible tracking
artifact* alongside the "bad title/snippet" explanation — both are named per-row below.

**#1 — content_44f34c0a90047651** (top_3, pos 0.67, CTR 0.011%, 212,404 impressions) — highest
score by raw opportunity size: the single largest impression volume in the top 20, well below
top_3's 0.35% baseline. **Would be wrong if:** this keyword's intent is informational/navigational
(searchers not expecting to click through) rather than a genuine snippet problem.

**#2 — content_8e1334d6356668e3** (top_3, pos 2.69, CTR 0.001%, 134,984 impressions) — huge
volume, near-zero CTR. **Would be wrong if:** the SERP result is dominated by a featured
snippet/People-Also-Ask box pulling the answer without a click, which no title rewrite fixes.

**#3 — content_fec55986a1868d62** (top_3, pos 0.31 — essentially rank #1, CTR 0.001%, 124,075
impressions) — position this strong with CTR this low is the tracking-artifact pattern flagged
above. **Would be wrong if:** this is a GSC/GA4 attribution gap (e.g. clicks landing on a
redirected URL not credited to this content_id) rather than a real snippet issue — worth a
manual GSC check before assuming a content fix is needed.

**#4 — content_9c057b66c30a3abb** (top_3, pos 0.12 — rank #1, CTR 0.001%, 83,834 impressions) —
same tracking-artifact pattern as #3, even stronger position. **Would be wrong if:** same as
#3 — verify real click data in GSC directly before treating this as a content problem.

**#5 — content_bf078007df823490** (top_3, pos 1.40, CTR literally 0.0%, 44,707 impressions,
only 20 active days) — zero clicks AND fewer active days than most peers (20 vs 29-31).
**Would be wrong if:** this page only recently entered the top_3 tier (hence fewer active
days) and simply hasn't accumulated clicks yet — a timing artifact, not a real CTR problem.

**#6 — content_d61fc394d10cba41** (top_3, pos 2.36, CTR 0.003%, 38,000 impressions) — solid
mid-pack quick-win candidate, no anomalies. **Would be wrong if:** the target keyword has
naturally low commercial intent (e.g. very broad informational query).

**#7 — content_fc67675904376267** (top_3, pos 2.13, CTR 0.030% — closest to baseline of the
top 10, 60,172 impressions) — flagged mainly on volume rather than an extreme CTR gap.
**Would be wrong if:** 0.030% is actually within normal range for this specific query type
and the "gap" is mostly a volume-weighting artifact, not a real underperformance.

**#8 — content_dc91779c3d085398** (top_3, pos 2.39, CTR 0.004%, 25,625 impressions) —
straightforward quick-win pattern, no anomalies. **Would be wrong if:** recent algorithm
volatility moved this page into top_3 only briefly and CTR hasn't caught up yet.

**#9 — content_66bf45eb0c5bb550** (top_3, pos 2.07, CTR 0.004%, 24,259 impressions) — same
pattern as #8. **Would be wrong if:** same timing/volatility explanation as #8.

**#10 — content_306bc78dff1eb683** (top_3, pos 1.44, CTR 0.043% — well above several peers,
80,821 impressions) — flagged mainly because of very high impression volume outweighing a
moderate CTR gap. **Would be wrong if:** 0.043% CTR is already reasonable for this query and
the ranking is overweighting raw volume via the `log1p` term.

**#11 — content_b9acd1ebff7d34ff** (top_3, pos 2.09, CTR 0.012%, 25,941 impressions) —
unremarkable, fits the core pattern cleanly. **Would be wrong if:** nothing stands out; lowest
individual risk of the top 20 for a false positive.

**#12 — content_757b1fa67827358d** (top_3, pos 2.86 — weakest position in the top 20, CTR
0.019%, 31,575 impressions) — this is the page closest to falling out of top_3 into striking.
**Would be wrong if:** its position is trending downward already, in which case CTR fix effort
is better spent elsewhere before this page drops tiers entirely.

**#13 — content_1d7764b642f7bb9f** (top_3, pos 1.12, CTR 0.017%, 23,402 impressions) —
unremarkable, fits the core pattern. **Would be wrong if:** no specific concern beyond the
general caveat that CTR gap alone doesn't diagnose the actual title/meta problem.

**#14 — content_aa99b9d554168e04** (top_3, pos 2.49, CTR 0.018%, 21,794 impressions) — same as
#13, unremarkable fit.

**#15 — content_fa17add7836d36c3** (top_3, pos 1.14, CTR 0.0%, 12,588 impressions) — zero CTR
at a strong position, smaller volume than #3/#4. **Would be wrong if:** same tracking-artifact
concern as #3/#4, though lower priority given the smaller impression base.

**#16 — content_b154f6c2652cfeb9** (top_3, pos 2.01, CTR 0.0%, 11,344 impressions) — zero CTR,
moderate position. **Would be wrong if:** same tracking-artifact concern, lower priority.

**#17 — content_cd3d932d4e1c8db0** (page_1, pos 7.83, CTR 0.004%, 89,332 impressions) — the
only non-top_3 row in the top 20, and it earns its place almost entirely on raw volume
(89,332 impressions, the 3rd-highest in the whole top 20). **Would be wrong if:** page_1's
lower baseline (0.29%) makes the "gap" look more dramatic than it really is relative to
what's achievable that far down the page — worth checking this isn't over-prioritized purely
by the log1p(impressions) term.

**#18 — content_f5a7a2559d483a54** (top_3, pos 2.96 — right at the tier boundary, CTR 0.0%,
10,886 impressions) — zero CTR at the weakest position in the entire top 20. **Would be
wrong if:** this page is barely holding top_3 and about to drop to striking — a ranking
problem, not a CTR problem, and the wrong lever to pull.

**#19 — content_c46df0fa61530d86** (top_3, pos 0.97, CTR 0.060% — highest CTR in the entire
top 20, 70,398 impressions) — flagged almost purely on volume, since its CTR is actually
decent. **Would be wrong if:** this page doesn't need a fix at all — its real problem, if any,
is elsewhere, and it's only in the queue because of impression size.

**#20 — content_520e203a08cd69ee** (top_3, pos 0.23 — essentially rank #1, CTR 0.0%,
10,462 impressions, only 22 active days) — same tracking-artifact concern as #3/#4/#5, plus
fewer active days than most peers. **Would be wrong if:** either the attribution gap explanation
(like #3/#4) or the recent-tier-entry explanation (like #5) — smallest impression base of the
zero-CTR cluster, so lowest-priority to actually investigate first.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

**Weakest picks, and why:**

1. **The zero-CTR-at-near-rank-#1 cluster (#3, #4, #15, #16, #18, #20)** — six of the top 20
   combine an extremely strong position (avg position under 3, several under 1.0) with
   literally 0% CTR. This is the single biggest risk in the queue: a page ranking #1 with
   genuinely zero clicks across tens of thousands of impressions is unusual enough that
   **attribution/tracking failure is at least as plausible as a real content problem** — e.g.
   clicks landing on a redirected URL not credited to this `content_hash_id`. If that's the
   case, no title/meta rewrite would fix anything, because the content isn't actually the
   problem. This should be manually spot-checked in GSC directly before an editor spends time
   on any of these six.

2. **#17 and #19 — flagged mostly by volume, not by a real CTR gap.** #19 has the highest CTR
   in the entire top 20 (0.060%) and is arguably not underperforming at all; it's in the queue
   because 70,398 impressions inflates its score via the `log1p` term. #17 is the only
   non-top_3 row, earning its spot almost entirely on its 89,332 impressions rather than a
   dramatic CTR gap relative to page_1's (already low) 0.29% baseline. Both suggest the score
   formula can overweight raw volume relative to how bad the actual CTR problem is — worth
   knowing before treating rank order as a strict priority order.

3. **#12 — a tier-boundary risk, not a CTR risk.** Weakest position in the top 20 (2.86, closest
   to falling into `striking`). If its position is already trending down, the real intervention
   needed is different from a CTR fix, and this rule has no way to see that trend (single-month
   snapshot, no trajectory signal).

**Leakage check.** Confirming the rule and score use only March-observed features, with no
future-window or product/decision-flag inputs:

In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Leakage check 1: confirm no April/future-window columns exist anywhere in the queue
future_window_terms = ["apr_", "may_", "jun_", "future", "next_month"]
leaked_cols = [c for c in queue.columns if any(t in c.lower() for t in future_window_terms)]
print(f"future-window columns present in queue: {leaked_cols if leaked_cols else 'none — clean'}")

# Leakage check 2: confirm none of the product/decision-flag columns identified in ML-05's
# Attack C (is_published, is_deleted, optimization_eligible_date, last_optimized_date) were used
PRODUCT_FLAGS = ["is_published", "is_deleted", "optimization_eligible_date", "last_optimized_date"]
leaked_flags = [c for c in queue.columns if c in PRODUCT_FLAGS]
print(f"product/decision-flag columns present in queue: {leaked_flags if leaked_flags else 'none — clean'}")

# Leakage check 3: confirm the scoring formula itself only references March-derived fields
score_inputs = ["position_tier", "gsc_ctr_x100", "gsc_impressions_total"]
print(f"\nscore formula inputs: {score_inputs}")
print("all three are computed from `mar` (March 2026) only — none touch `apr`, `dim_content`'s "
      "flagged columns, or `content_updated_date` (excluded per ML-06's synthetic-date finding).")

assert not leaked_cols, "future-window column found in queue!"
assert not leaked_flags, "product/decision flag found in queue!"
print("\nleakage check passed — no future-window or product-flag inputs in the ranked queue.")

future-window columns present in queue: none — clean
product/decision-flag columns present in queue: none — clean

score formula inputs: ['position_tier', 'gsc_ctr_x100', 'gsc_impressions_total']
all three are computed from `mar` (March 2026) only — none touch `apr`, `dim_content`'s flagged columns, or `content_updated_date` (excluded per ML-06's synthetic-date finding).

leakage check passed — no future-window or product-flag inputs in the ranked queue.


**Result:** no future-window columns and none of ML-05's flagged product/decision fields
appear anywhere in the queue or the scoring formula — confirmed programmatically, not just
asserted. The score depends only on `position_tier`, `gsc_ctr_x100`, and
`gsc_impressions_total`, all computed exclusively from March 2026 GSC data.

## Self-check

Before you submit, confirm each line honestly:

- [✅] Every section above is filled — markdown thinking AND the code that backs it
- [✅] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✅] No client names, URLs, or private queries anywhere
- [✅] My claims use careful words: observed, measured, directional, decision-support
- [✅] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.